# IO.Inventory Functions

There are three main locations or GOES MCIMP DataProducts, locally on current device, on AWS in an S3 bucket and stored in a THREDDS catalog. The functions in this section are used to inspect files, list files, count files, etc.

In [2]:
# Getting a list of files from a THREDDS catalog

from siphon.catalog import TDSCatalog

ModuleNotFoundError: No module named 'siphon'

In [ ]:
url = 'https://redoak.cs.toronto.edu/twitcher/ows/proxy/thredds/catalog/datasets/GOES/GOES18/ABI-L2-MCMIPF/2022/209/00/catalog.html?dataset=datasets/GOES/GOES18/ABI-L2-MCMIPF/2022/209/00/OR_ABI-L2-MCMIPF-M6_G18_s20222090000172_e20222090009486_c20222090009581.nc'

cat = TDSCatalog(url)

In [ ]:
help(cat)

In [ ]:
cat.catalog_refs

In [ ]:
cat.base_tds_url

In [ ]:
cat.catalog_url

In [ ]:
cat.datasets

In [ ]:
cat.datasets[0].remote_access(use_xarray=True)

In [ ]:
cat.ds_with_access_elements_to_process

In [ ]:
cat.latest

In [ ]:
cat.metadata

In [ ]:
cat.services

In [ ]:
cat.session

In [44]:
import warnings
from siphon.catalog import TDSCatalog
from urllib.parse import urlparse
import csv

def walk_thredds_catalog(url, csv_path, row_accumulator=None):
    """
    Walk a THREDDS catalog recursively and save leaf-hour-level catalogs to CSV.
    Suppresses the HTML-to-XML redirect warning from Siphon.
    """
    if row_accumulator is None:
        row_accumulator = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        cat = TDSCatalog(url)

    if cat.catalog_refs:  # non-leaf, recurse
        for subdir in cat.catalog_refs:
            base, last = url.rsplit('/', 1)  # split at last segment (catalog.xml)
            subcat_url = f"{base}/{subdir}/{last}"  # construct child catalog URL
            walk_thredds_catalog(subcat_url, csv_path, row_accumulator)
    else:  # leaf node
        # Parse segments from URL
        parts = urlparse(url).path.split('/')
        try:
            satellite = parts[-6]
            year = parts[-4]
            day_of_year = parts[-3]
            hour = parts[-2]
            row_accumulator.append([satellite, year, day_of_year, hour, url])
        except IndexError:
            row_accumulator.append([None]*5 + [url])

    # Only write CSV at the top-level call
    if csv_path and row_accumulator:
        with open(csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['satellite', 'year', 'day_of_year', 'hour', 'hour_url'])
            writer.writerows(row_accumulator)

    return len(row_accumulator)


In [45]:
base_url = 'https://redoak.cs.toronto.edu/twitcher/ows/proxy/thredds/catalog/datasets/GOES/GOES16/ABI-L2-MCMIPF/2021/131/catalog.html'
total_hours = walk_thredds_catalog(base_url, 'goes18_catalog.csv')
print(f"Total hour-level catalogs: {total_hours}")


Total hour-level catalogs: 24


In [47]:
goes_16 = 'https://redoak.cs.toronto.edu/twitcher/ows/proxy/thredds/catalog/datasets/GOES/GOES16/catalog.html'
goes_18 = 'https://redoak.cs.toronto.edu/twitcher/ows/proxy/thredds/catalog/datasets/GOES/GOES18/catalog.html'
goes_19 = 'https://redoak.cs.toronto.edu/twitcher/ows/proxy/thredds/catalog/datasets/GOES/GOES19/catalog.html'

In [50]:
%%time
goes_19_hours = walk_thredds_catalog(goes_19, 'goes19_catalog.csv')
print(f"Total hour-level catalogs: {goes_19_hours}")

Total hour-level catalogs: 7542
CPU times: total: 28min 1s
Wall time: 1h 17s
